# Truss + vLLM on AzureML — Build, Test & Publish a *Faithful* Container

## Goal of this exercise
Prove that a **Truss-packaged vLLM model** can run on an **AzureML managed online endpoint** *faithfully* — honoring the container's **own declared serving contract** — and package it so an AzureML **Deployment Template (DT)** can deploy it.

**"Faithful" means all three of:**
1. **Truss's Dockerfile is used UNMODIFIED** (zero-edit). The only file we author is `config.yaml` (the Truss *package descriptor*) — never the generated container definition.
2. The model serves **native OpenAI `POST /v1/chat/completions`** on the port Truss declares (`server_port: 8000`), exactly as Baseten's platform would route it.
3. Model **weights are MOUNTED** at runtime from the AzureML model (not baked into the image).

## Why (the bigger picture)
This validates **Baseten ↔ AzureML** integration for the Model Catalog. A Truss `docker_server` package declares `server_port`, `predict_endpoint`, `readiness/liveness_endpoint`, and mounted `weights`. Baseten's platform consumes those to expose an OpenAI endpoint; **AzureML's `inference_config` — surfaced through a Deployment Template — is the exact equivalent knob.** The integration is a mechanical field mapping:

| Truss `config.yaml` (`docker_server`) | AzureML Deployment Template |
|---|---|
| `server_port: 8000` | `scoring_port: 8000` + probe `port` |
| `predict_endpoint: /v1/chat/completions` | `scoring_path` |
| `readiness_endpoint: /health` | `readiness_probe.path` |
| `liveness_endpoint: /health` | `liveness_probe.path` |
| mounted weights | `model_mount_path` + registered model |

## What this notebook does
- **Phase 0–1** — Preflight + configuration (auto-detected; no secrets committed).
- **Phase 2** — Generate the Truss build context and inspect the **UNMODIFIED** Dockerfile.
- **Phase 3** — Build the image on this A100 box (native amd64, no emulation).
- **Phase 4–5** — Download weights, run the container **on the GPU**, and validate `/v1/chat/completions` **before** publishing.
- **Phase 6** — Push the validated image to the workspace ACR.
- **Phase 7** — Create the AzureML **workspace** environment (image-based).
- **Phase 8** — **Promote** the environment to the **registry** (so a registry DT can consume it).
- **Phase 9** — (Guided) Deploy via a registry DT and validate on a managed endpoint.
- **Phase 10** — Stop the compute instance to save cost.

## Where this runs
On the **AzureML A100 compute instance `truss-build-a100`** (`Standard_NC24ads_A100_v4`, A100 80 GB) — it has Docker **and** a GPU, which the build-and-test loop needs. Open this notebook there, with the repo cloned on the CI.


## Prerequisites — read before running
- You are running this **on the `truss-build-a100` compute instance** (not your laptop): it has Docker + the A100 GPU.
- The repo is **cloned on the CI** and you opened this notebook from `truss-poc/vllm/notebooks/`.
- `az` is logged in — AzureML compute instances authenticate as you automatically (verified in Phase 0).
- Set your **registry** once before launching Jupyter (or edit the config cell):
  ```bash
  export AZUREML_REGISTRY=<your-registry>
  ```

> **No secrets live in this notebook.** Subscription is read from `az`, workspace + resource group from the CI's AzureML config, and the registry from an environment variable.


## Phase 0 — Preflight: GPU, Docker, disk, `az`
Confirms this box has everything the build-and-test loop needs. If any check fails, stop and fix it before continuing.


In [ ]:
import subprocess

def run(cmd):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, text=True).returncode

print("=== GPU ===")
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || echo 'NO GPU — are you on the A100 CI?'")
print("\n=== Docker daemon ===")
run("docker version --format 'client={{.Client.Version}} server={{.Server.Version}}' || echo 'NO DOCKER'")
print("\n=== Docker can reach the GPU (pulls a small CUDA base once) ===")
run("docker run --rm --gpus all nvidia/cuda:12.4.0-base-ubuntu22.04 nvidia-smi -L || echo 'GPU-in-docker check failed (revisit before Phase 5)'")
print("\n=== Free disk (need ~30 GB for the vLLM image) ===")
run("df -h / /mnt 2>/dev/null | awk 'NR==1 || /\\/mnt|\\/$/'")
print("\n=== az identity ===")
run("az account show --query '{subscription:id, user:user.name}' -o yaml || echo 'RUN: az login'")


## Phase 1 — Configuration (auto-detected, no secrets)
- **Subscription** — from your `az` login.
- **Workspace + resource group** — from the CI's AzureML config (`~/.azureml/config.json`), or `RESOURCE_GROUP` / `AZUREML_WORKSPACE` env vars.
- **Registry** — from the `AZUREML_REGISTRY` env var (set it if the check below complains — it is not auto-detectable).

Everything else is a fixed **asset name** for this PoC. Run this cell first; it's referenced by every later phase.


In [ ]:
import os, json, glob, subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

# --- Subscription: from az (never hardcoded) ---
SUBSCRIPTION_ID = os.environ.get("SUBSCRIPTION_ID") or sh("az account show --query id -o tsv")

# --- Workspace + RG: from the CI's AzureML config, or env ---
cfg = {}
for p in ["/home/azureuser/.azureml/config.json", os.path.expanduser("~/.azureml/config.json")]:
    if os.path.exists(p):
        try: cfg = json.load(open(p))
        except Exception: pass
        break
RESOURCE_GROUP = os.environ.get("RESOURCE_GROUP") or cfg.get("resource_group", "")
WORKSPACE      = os.environ.get("AZUREML_WORKSPACE") or cfg.get("workspace_name", "")

# --- Registry: set via env (not auto-detectable) ---
REGISTRY = os.environ.get("AZUREML_REGISTRY", "")

# --- Fixed asset names for this PoC ---
MODEL_NAME, MODEL_VERSION = "truss-vllm-qwen35", "1"
ENV_NAME,   ENV_VERSION   = "truss-vllm-server", "3"
IMG                       = f"{ENV_NAME}:{ENV_VERSION}"       # local docker tag
INSTANCE_TYPE             = "Standard_NC24ads_A100_v4"
CI_NAME                   = "truss-build-a100"

# --- Repo paths ---
REPO_ROOT       = sh("git rev-parse --show-toplevel") or os.getcwd()
TRUSS_MODEL_DIR = os.path.join(REPO_ROOT, "truss-poc/vllm/truss-model")
BUILD_CTX       = os.path.join(REPO_ROOT, "truss-poc/vllm/.build-context")
MODEL_DL        = os.path.join(REPO_ROOT, "truss-poc/vllm/.model")

print(f"SUBSCRIPTION_ID = {SUBSCRIPTION_ID or '❌ run az login'}")
print(f"RESOURCE_GROUP  = {RESOURCE_GROUP or '⚠️  export RESOURCE_GROUP=...'}")
print(f"WORKSPACE       = {WORKSPACE or '⚠️  export AZUREML_WORKSPACE=...'}")
print(f"REGISTRY        = {REGISTRY or '⚠️  export AZUREML_REGISTRY=...'}")
print(f"REPO_ROOT       = {REPO_ROOT}")
assert SUBSCRIPTION_ID and RESOURCE_GROUP and WORKSPACE, "Set the missing values above, then re-run."
print("\n✅ config ready")


## Phase 2 — Generate the Truss build context (UNMODIFIED Dockerfile)
Truss's `docker_server` backend turns `config.yaml` into a container build context (Dockerfile + nginx + supervisord, wrapping `vllm/vllm-openai`). **We do not edit the emitted Dockerfile.** The only thing that declares behavior is `config.yaml`, whose inline `start_command` launches vLLM against the mounted weights:

```yaml
docker_server:
  start_command: vllm serve /opt/ml/model --served-model-name model --host 0.0.0.0 --port 8000 ...
  server_port: 8000
  predict_endpoint: /v1/chat/completions
  readiness_endpoint: /health
  liveness_endpoint: /health
```

We print the Dockerfile so you can see (a) it's stock Truss, and (b) the one Azure-relevant wrinkle: Truss rewrites apt to a community mirror (`mirror://mirrors.ubuntu.com`) that can be flaky *inside Azure's build service*. We build here — on a normal CI NIC — precisely so that mirror resolves without patching.


In [ ]:
# Pin the exact Truss version this flow was validated with
subprocess.run("pip install -q 'truss==0.18.20'", shell=True, check=True)

subprocess.run(f"rm -rf {BUILD_CTX}", shell=True)
subprocess.run(f"truss image build-context {BUILD_CTX} {TRUSS_MODEL_DIR} --non-interactive",
               shell=True, check=True)

print("=== Emitted Dockerfile (UNMODIFIED — we never patch it) ===\n")
print(open(os.path.join(BUILD_CTX, "Dockerfile")).read())
print("\n=== supervisord.conf (model-server = our inline start_command; nginx kept but idle on AzureML) ===\n")
print(open(os.path.join(BUILD_CTX, "supervisord.conf")).read())


## Phase 3 — Build the image (native amd64 on the A100 box)
Building here avoids a laptop's slow arm64 emulation **and** the ~10 GB push over home internet (the CI pushes to ACR intra-Azure).

**If `apt-get install nginx` fails** with a mirror 404, that's the known Azure↔community-mirror flakiness. Fallbacks, in order:
1. **Re-run this cell** — the mirror method picks a random mirror each time.
2. If it persists, run the clearly-marked **apt-mirror revert** cell below, then re-run this build. That revert is the *only* place we would ever touch the Dockerfile, and it changes *where apt downloads from* — not the container's behavior.


In [ ]:
rc = subprocess.run(f"cd {BUILD_CTX} && docker build -t {IMG} .", shell=True).returncode
print("\n✅ BUILD OK" if rc == 0 else "\n❌ BUILD FAILED — see the apt-mirror fallback cell below, then re-run this cell")


### (Only if Phase 3 failed on the apt mirror) — one-line, behavior-neutral revert
Reverts Truss's apt mirror to the canonical Ubuntu archive, then you re-run Phase 3. This is the sole Dockerfile edit we ever make, and only as a fallback.


In [ ]:
# UNCOMMENT the 4 lines below only if Phase 3 failed on 'apt-get install nginx':
# dfp = os.path.join(BUILD_CTX, "Dockerfile")
# s = open(dfp).read().replace("mirror://mirrors.ubuntu.com/US.txt", "http://archive.ubuntu.com/ubuntu/")
# open(dfp, "w").write(s)
# print("Reverted apt mirror to archive.ubuntu.com. Now re-run Phase 3.")


## Phase 4 — Download the model weights (to MOUNT, not bake)
We simulate AzureML's runtime mount by downloading the registered model and mounting it at `/opt/ml/model` when we run the container. vLLM's `--model /opt/ml/model` expects `config.json` at that root, so we locate the exact directory that contains it.


In [ ]:
subprocess.run(f"rm -rf {MODEL_DL} && mkdir -p {MODEL_DL}", shell=True)
subprocess.run(
    f"az ml model download --name {MODEL_NAME} --version {MODEL_VERSION} "
    f"--download-path {MODEL_DL} --resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE}",
    shell=True, check=True)

cfgs = glob.glob(f"{MODEL_DL}/**/config.json", recursive=True)
assert cfgs, f"config.json not found under {MODEL_DL}"
WEIGHTS_DIR = os.path.dirname(cfgs[0])
print("WEIGHTS_DIR =", WEIGHTS_DIR)
subprocess.run(f"ls -la {WEIGHTS_DIR} | head", shell=True)


## Phase 5 — Run on the GPU and validate `/v1/chat/completions`
The crux of the exercise: run the **exact image we'll publish**, on a real GPU, with weights mounted at `/opt/ml/model` (mirroring the DT's `model_mount_path`), and confirm:
- `GET /health` → 200 — this is what AzureML's probes hit.
- `POST /v1/chat/completions` → an OpenAI `chat.completion` — native, streaming-capable.
- Truss's nginx path `POST /v1/models/model:predict` on `:8080` → also chat completions — proving the container honors the **full Truss contract** too.


In [ ]:
import time, json, urllib.request

subprocess.run("docker rm -f truss-vllm-test 2>/dev/null", shell=True)
subprocess.run(
    f'docker run -d --name truss-vllm-test --gpus all '
    f'-v "{WEIGHTS_DIR}":/opt/ml/model -p 8000:8000 -p 8080:8080 {IMG}',
    shell=True, check=True)

def get(url):
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            return r.status
    except Exception:
        return None

print("Waiting for vLLM to load the model...")
healthy = False
for i in range(72):  # up to ~6 min
    if get("http://localhost:8000/health") == 200:
        healthy = True; print(f"/health → 200 after ~{i*5}s"); break
    time.sleep(5)

if not healthy:
    print("❌ vLLM did not become healthy — container logs:")
    subprocess.run("docker logs --tail 80 truss-vllm-test", shell=True)
    raise SystemExit("startup failed")


In [ ]:
# Native OpenAI chat completions — directly against vLLM on :8000 (the declared server_port)
req = json.dumps({
    "model": "model",
    "messages": [{"role": "user", "content": "What is the capital of France?"}],
    "max_tokens": 30, "temperature": 0.1,
}).encode()

r = urllib.request.Request("http://localhost:8000/v1/chat/completions",
                           data=req, headers={"Content-Type": "application/json"})
with urllib.request.urlopen(r, timeout=90) as resp:
    body = json.load(resp)

print("=== POST /v1/chat/completions (vLLM :8000) ===")
print(json.dumps(body, indent=2)[:900])
assert body.get("object") == "chat.completion", "unexpected response shape"
print("\n✅ Native OpenAI chat completions works on the declared contract (:8000)")


In [ ]:
# (Optional) Truss's nginx on :8080 rewrites /v1/models/model:predict -> /v1/chat/completions
r = urllib.request.Request("http://localhost:8080/v1/models/model:predict",
                           data=req, headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(r, timeout=90) as resp:
        b = json.load(resp)
    print("nginx :8080 /v1/models/model:predict →", b.get("object"))
    print("✅ Container also honors Truss's native predict route")
except Exception as e:
    print("nginx path check (non-fatal):", e)


## Phase 6 — Push the validated image to the workspace ACR
Now that it serves correctly, push it. The CI → ACR push is intra-Azure (fast). We resolve the workspace's linked ACR, log in, tag, and push.


In [ ]:
ACR_ID = subprocess.run(
    f"az ml workspace show --name {WORKSPACE} --resource-group {RESOURCE_GROUP} "
    f"--query container_registry -o tsv", shell=True, capture_output=True, text=True).stdout.strip()
ACR_NAME = ACR_ID.split("/")[-1]
ACR_IMG  = f"{ACR_NAME}.azurecr.io/{ENV_NAME}:{ENV_VERSION}"
print("workspace ACR:", ACR_NAME)

subprocess.run(f"az acr login --name {ACR_NAME}", shell=True, check=True)
subprocess.run(f"docker tag {IMG} {ACR_IMG}", shell=True, check=True)
subprocess.run(f"docker push {ACR_IMG}", shell=True, check=True)
print("\n✅ pushed:", ACR_IMG)


## Phase 7 — Create the AzureML **workspace** environment (image-based)
Register an environment that points at the pushed image (`image:` — no build step). This is the "publish to a workspace env" step.


In [ ]:
env_yaml = f'''$schema: https://azuremlschemas.azureedge.net/latest/environment.schema.json
name: {ENV_NAME}
version: "{ENV_VERSION}"
image: {ACR_IMG}
description: "Truss docker_server + vLLM (Qwen3.5-0.8B) — prebuilt image, mounted weights"
'''
open("/tmp/env.image.yml", "w").write(env_yaml)
subprocess.run(
    f"az ml environment create --file /tmp/env.image.yml "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE}",
    shell=True, check=True)
print(f"\n✅ workspace environment created: {ENV_NAME}:{ENV_VERSION}")


## Phase 8 — **Promote** the environment to the registry
A Deployment Template must reference a **registry** environment (not a workspace one). `az ml environment share` copies the environment — and its image — from the workspace into the registry, so the registry DT can consume it.


In [ ]:
subprocess.run(
    f"az ml environment share --name {ENV_NAME} --version {ENV_VERSION} "
    f"--resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE} "
    f"--registry-name {REGISTRY} --share-with-name {ENV_NAME} --share-with-version {ENV_VERSION}",
    shell=True, check=True)
print(f"\n✅ shared to registry '{REGISTRY}'")
subprocess.run(
    f"az ml environment show --name {ENV_NAME} --version {ENV_VERSION} "
    f"--registry-name {REGISTRY} --query '{{name:name, image:image}}' -o yaml", shell=True)


## Phase 9 (guided) — Deploy via a registry DT and validate on an endpoint
This closes the loop. The promoted registry environment `truss-vllm-server:3` is now consumable by a Deployment Template. To deploy **faithfully**, the DT must mirror `config.yaml`:

| DT field | Value | Mirrors `config.yaml` |
|---|---|---|
| `environment` | `azureml://registries/<your-registry>/environments/truss-vllm-server/versions/3` | the promoted image |
| `scoring_port` | `8000` | `server_port` |
| `scoring_path` | `/v1/chat/completions` | `predict_endpoint` |
| `liveness_probe.path` / `port` | `/health` / `8000` | `liveness_endpoint` |
| `readiness_probe.path` / `port` | `/health` / `8000` | `readiness_endpoint` |
| `model_mount_path` | `/opt/ml/model` | mounted weights |

Use the repo's tested scripts to create the DT + endpoint + deployment (they handle the AOT model manifest and the DT registry PATCH):

```bash
cd truss-poc/vllm/scripts
export AZUREML_REGISTRY=<your-registry>   # + RESOURCE_GROUP / AZUREML_WORKSPACE if not on the CI
# Edit yaml/deployment-template.yml to scoring_port 8000, probes /health:8000, env version 3, then:
bash 3-create-deployment-template.sh    # registers/updates the registry DT (env v3)
bash 4-create-endpoint.sh               # recreates the endpoint (was deleted to save GPU)
bash 5-create-deployment.sh             # deploys via the DT (A100)
bash 6-route-traffic.sh
bash 7-test-inference.sh                 # calls /v1/chat/completions on the endpoint
```

Then validate with the OpenAI SDK against the endpoint:
```python
from openai import OpenAI
client = OpenAI(base_url=f"{scoring_uri.rsplit('/score',1)[0]}/v1", api_key=primary_key)
client.chat.completions.create(model="model",
    messages=[{"role":"user","content":"Capital of France?"}], max_tokens=30)
```


## Phase 10 — Stop the compute instance (save cost)
The A100 bills while running. Stop it when you're done (disk persists across stop/start). The cell below removes the local test container and prints the stop command.


In [ ]:
subprocess.run("docker rm -f truss-vllm-test 2>/dev/null", shell=True)
print("removed local test container.")
print(f"\nStop the A100 CI when done:")
print(f"  az ml compute stop --name {CI_NAME} --resource-group {RESOURCE_GROUP} --workspace-name {WORKSPACE}")
